# RPO

- Arbitrary RAAN, inclination, and phasing
- Angles, then 3 DOF, then 6 DOF

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import numpy as np
from cesium.constants import DEG_TO_RAD, KM_TO_M, M_TO_KM, RAD_TO_DEG, SEC_TO_MIN
from cesium.guidance.coplanar import guid_coaxial_hohmann
from cesium.guidance.noncoplanar import guid_combined_i_raan_change
from cesium.math.base import angle_between, unit
from cesium.math.rotations import wrap_to_2pi
from cesium.orbital.coes import (
    COES,
    E_from_ta,
    kepler_eq,
    ta_from_E,
)
from cesium.orbital.transformations import calc_dcm_rsw_wrt_pci
from cesium.physics.dynamics import propagate_two_body
from cesium.plotting import plot_3d
from cesium.world.constants import MU_MARS, R_MARS
from numpy.linalg import norm

## Setup
Assume within 5 degrees inclination, raan, 0.05 e, 100km a

In [24]:
# Target orbit (same as arrival.ipynb), MCI, circular
PARKING_ORBIT_ALT_M = 300e3  # m
PARKING_ORBIT_RAAN_DEG = 15
PARKING_ORBIT_I_DEG = 45
PARKING_ORBIT_TA_DEG = 13

TARGET_COES0 = COES(
    a=R_MARS + PARKING_ORBIT_ALT_M,
    e=0.0,
    i=PARKING_ORBIT_I_DEG * DEG_TO_RAD,
    raan=PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
    aop=0.0,
    ta=PARKING_ORBIT_TA_DEG * DEG_TO_RAD,
)
a_t, e_t, i_t, raan_t, aop_t, ta_t = TARGET_COES0

In [25]:
# Chaser = target + seeded uniform dispersion
SEED = 525600
ANGLE_DISP_DEG = 3.0  # i, raan, aop
A_DISP_M = 5 * KM_TO_M
E_MAX = 0.01  # e >= 0, so draw from [0, E_MAX]
INITIAL_TA_OFFSET_DEG = 22

rng = np.random.default_rng(SEED)
rel_coes = np.array(
    [
        rng.uniform(-A_DISP_M, A_DISP_M),
        rng.uniform(0.0, E_MAX),
        *rng.uniform(-ANGLE_DISP_DEG, ANGLE_DISP_DEG, 3) * DEG_TO_RAD,
        INITIAL_TA_OFFSET_DEG * DEG_TO_RAD,  # ta: phasing left to the RPO problem
    ]
)
CHASER_COES0 = COES.from_array(TARGET_COES0.elements() + rel_coes)
a_c, e_c, i_c, raan_c, aop_c, ta_c = CHASER_COES0

TARGET_COES0.print("initial target coes")
CHASER_COES0.print("initial chaser coes")

# mean motion
n_c0 = np.sqrt(MU_MARS / a_c**3)
n_t0 = np.sqrt(MU_MARS / a_t**3)

initial target coes: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 13.000 deg
initial chaser coes: a = 3698.946 km, e = 0.00763, i = 46.710 deg, raan = 15.208 deg, aop = 2.930 deg, ta = 35.000 deg


In [26]:
# Propagate both for a few target periods
N_ORBITS = 3
period_s = 2 * np.pi * np.sqrt(a_t**3 / MU_MARS)
t_s = np.linspace(0, N_ORBITS * period_s, 3000)

rv_target_mci = propagate_two_body(
    np.concatenate(TARGET_COES0.to_rv(MU_MARS)), t_s, mu=MU_MARS
)
rv_chaser0 = np.concatenate(CHASER_COES0.to_rv(MU_MARS))
rv_chaser_mci = propagate_two_body(rv_chaser0, t_s, mu=MU_MARS)

In [27]:
t0 = 0

## Planning
- long way
  - raan and i
  - circularize
  - phasing
- short way
  - minimum combined plane change (but figure that out)

In [28]:
# Estimation of dV cost

# Correcting raan and i
dv_i_raan = guid_combined_i_raan_change(
    i_c,
    i_t,
    raan_t - raan_c,
    norm(rv_chaser0[3:6]),
)
print(f"dv1 = {dv_i_raan:.3f} m/s")

# a * (1 - e)
r_p_chaser = a_c * (1 - e_c)
r_a_chaser = a_c * (1 + e_c)
r_p_target = a_t * (1 - e_t)
r_a_target = a_t * (1 + e_t)
assert r_p_target == r_a_target

# Circularizing
a_trans, t_trans, dv_a, dv_b = guid_coaxial_hohmann(
    r_p_chaser, r_a_chaser, r_p_target, r_a_target, mu=MU_MARS
)
print(f"dv2 = {dv_a:.3f} m/s")
print(f"dv3 = {dv_b:.3f} m/s")
print(f"Transfer time: {t_trans * SEC_TO_MIN:.3f}m")

dv1 = 102.595 m/s
dv2 = -7.149 m/s
dv3 = 5.843 m/s
Transfer time: 56.559m


**Larger plan**
1. Line of nodes
2. Find time until that node
3. Burn at `t1` at that node in specified direction for `i` and `raan` change
4. Recalculate new coes

### Burn 1

In [29]:
# Find node line
r_c0, v_c0 = rv_chaser_mci[0, 0:3], rv_chaser_mci[0, 3:6]

h_chaser = np.cross(r_c0, v_c0)
h_target = np.cross(rv_target_mci[0, 0:3], rv_target_mci[0, 3:6])
n_hat = unit(np.cross(h_chaser, h_target))

# Get times where chaser crosses that line
d_ta = angle_between(r_c0, n_hat) * np.sign(
    np.dot(v_c0, n_hat)
)  # change sign if further in orbit
d_ta = wrap_to_2pi(d_ta)
print(f"Angle until node line: {d_ta * RAD_TO_DEG}º")

Angle until node line: 146.9924342775553º


In [30]:
# Target position at the burn
def M_from_ta(ta,e):
    E = E_from_ta(ta, e)
    M = E - e*np.sin(E)
    return M

def ta_from_M(M,e):
    E = kepler_eq(M,e)
    ta = ta_from_E(E,e)
    return ta

In [31]:
M_c0 = M_from_ta(CHASER_COES0.ta, CHASER_COES0.e)

ta_c1 = ta_c + d_ta
M_c1 = M_from_ta(ta_c1, CHASER_COES0.e)

dM = wrap_to_2pi(M_c1 - M_c0)

t_coast = dM / CHASER_COES0.n(MU_MARS)
t1 = t_coast
print(f"Coast time: {t_coast * SEC_TO_MIN}")

Coast time: 46.64830405018724


In [32]:
M_t0 = M_from_ta(ta_t, e_t)
M_t1 = M_t0 + t_coast * TARGET_COES0.n(MU_MARS)
ta_t1 = ta_from_M(M_t1,e_t)
E_t1 = kepler_eq(M_t1, e_t)
ta_t1 = ta_from_E(E_t1, e_t)

TARGET_COES1 = TARGET_COES0.with_ta(ta_t1)
CHASER_COES1 = CHASER_COES0.with_ta(ta_c1)
r_t1, v_t1 = TARGET_COES1.to_rv(MU_MARS)
r_c1, v_c1 = CHASER_COES1.to_rv(MU_MARS)

In [33]:
vectors = [
    {
        "direction": n_hat,
        "length": 20000,
        "name": "Node line",
        "color": "goldenrod",
        "label": True,
    },
]

points = [
    {
        "name": "r_t1",
        "location": r_t1 * M_TO_KM,
        "color": "blue",
    },
    {
        "name": "r_c1",
        "location": r_c1 * M_TO_KM,
        "color": "yellow",
    },
    # {
    #     "name": "SOI along v_inf",
    #     "location": -mars_soi_km * unit(v_inf_mci_km),
    #     "color": "green",
    # },
]

fig = plot_3d(
    [rv_target_mci[:, 0:3] * M_TO_KM, rv_chaser_mci[:, 0:3] * M_TO_KM],
    [t_s, t_s],
    labels=["Target", "Chaser"],
    trace_colors=["purple", "green"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title=f"Target and chaser, {N_ORBITS} orbits (MCI)",
    vectors=vectors,
    points=points,
    units="km",
)

In [34]:
dv_burn1 = dv_i_raan * unit(h_chaser)
CHASER_COES1A = COES.from_rv(r_c1, v_c1 + dv_burn1, MU_MARS)

dv_burn1_rsw = calc_dcm_rsw_wrt_pci(np.concatenate((r_c1, v_c1))) @ dv_burn1
print(
    f"Burn 1 at t = {t_coast * SEC_TO_MIN:.2f} min (node, {d_ta * RAD_TO_DEG:.2f} deg down orbit from t0)"
)
print(
    f"  |dv| = {norm(dv_burn1):.3f} m/s; RSW = [{dv_burn1_rsw[0]:.3f}, {dv_burn1_rsw[1]:.3f}, {dv_burn1_rsw[2]:.3f}] m/s"
)
print(f"  speed: {norm(v_c1):.3f} -> {norm(v_c1 + dv_burn1):.3f} m/s")
print()
TARGET_COES1.print("target         ")
CHASER_COES1.print("chaser before  ")
CHASER_COES1A.print("chaser after   ")
CHASER_COES1A.delta(CHASER_COES1).print("change         ")
CHASER_COES1A.delta(TARGET_COES1).print("after - target ")

Burn 1 at t = 46.65 min (node, 146.99 deg down orbit from t0)
  |dv| = 102.595 m/s; RSW = [-0.000, -0.000, 102.595] m/s
  speed: 3376.871 -> 3378.429 m/s

target         : a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 160.698 deg
chaser before  : a = 3698.946 km, e = 0.00763, i = 46.710 deg, raan = 15.208 deg, aop = 2.930 deg, ta = 181.992 deg
chaser after   : a = 3702.311 km, e = 0.00671, i = 44.977 deg, raan = 14.997 deg, aop = 2.804 deg, ta = 182.265 deg
change         : a = 3.366 km, e = -0.00092, i = -1.734 deg, raan = -0.211 deg, aop = -0.126 deg, ta = 0.273 deg
after - target : a = 6.311 km, e = 0.00671, i = -0.023 deg, raan = -0.003 deg, aop = 2.804 deg, ta = 21.567 deg


### Burns 2 and 3

In [ ]:
# Recalculating dv
a_c, e_c = CHASER_COES1A.a, CHASER_COES1A.e
a_t, e_t = TARGET_COES1.a, TARGET_COES1.e # in case it is different?!

# Very circular, so we should just depart at whichever is sooner
periapsis_is_closer = CHASER_COES1A.ta > np.pi
print(f"{periapsis_is_closer = }")
a_trans, t_trans, dv_a, dv_b = guid_coaxial_hohmann(
    a_c*(1-e_c), a_c*(1-e_c), a_t*(1-e_t), a_t*(1+e_t), depart_at_periapsis=periapsis_is_closer, mu=MU_MARS
)
print(f"sma of transfer orbit: {a_trans} m")
print(f"Transfer orbit (half) period {t_trans}")
print(f"{dv_a = :.3f} m/s (negative means retrograde)")
print(f"{dv_b = :.3f} m/s (negative means retrograde)")

periapsis_is_closer = True
sma of transfer orbit: 3686731.0040687677 m
Transfer orbit (half) period 3398.193873067619
dv_a = 4.287 m/s (negative means retrograde)
dv_b = 4.282 m/s (negative means retrograde)


In [ ]:
M_burn2 = 2*np.pi if periapsis_is_closer else np.pi
M_c1a = M_from_ta(CHASER_COES1A.ta, CHASER_COES1A.e)
dt = (M_burn2 - M_c1a) / CHASER_COES1A.n(MU_MARS)
t2 = t1 + dt
M_c2 = wrap_to_2pi(M_burn2)
ta_c2 = ta_from_M(M_c2, CHASER_COES1A.e)

CHASER_COES2 = CHASER_COES1A.with_ta(ta_c2)
r_c2, v_c2 = CHASER_COES2.to_rv(MU_MARS)

M_t2 = wrap_to_2pi(M_t1 + dt * TARGET_COES1.n(MU_MARS))
ta_t2 = ta_from_M(M_t2, TARGET_COES1.e)
TARGET_COES2 = TARGET_COES1.with_ta(ta_t2)
r_t2, v_t2 = TARGET_COES2.to_rv(MU_MARS)

t_1_to_2 = np.linspace(t1, t2, 3000)
rv_chaser_1_to_2_mci = propagate_two_body(np.concatenate((r_c2, v_c2)), t_s, mu=MU_MARS)

In [45]:
t3 = t2 + t_trans

6175.0373637422945


In [ ]:
CHASER_COES2A = COES.from_rv(r_c2, v_c2 + dv_a*unit(v_c2))

## Visualization

In [39]:
vectors = [
    {
        "direction": n_hat,
        "length": 20000,
        "name": "Node line",
        "color": "goldenrod",
        "label": True,
    },
]

fig = plot_3d(
    [rv_target_mci[:, 0:3], rv_chaser_1_to_2_mci[:, 0:3] * M_TO_KM],
    [t_s, t_s],
    labels=["Target", "Chaser"],
    trace_colors=["black", "royalblue"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title=f"Target and chaser, {N_ORBITS} orbits (MCI)",
    vectors=vectors,
    units="km",
)